## 🧠 Sleep Blindspots — Epoch Dataset Construction

### Objective

Convert raw EEG signals and sleep stage annotations into a structured, epoch-level dataset for machine learning.

---

### Key Idea

Instead of working with continuous signals, we represent sleep as a sequence of fixed-length segments: 30-second EEG segment → one sleep stage label


---

### Why This Matters

Sleep stage classification is defined at the **epoch level**, not at the raw signal level.

This representation allows us to:

- Build ML models
- Compare stable vs transition regions
- Analyze failure modes

---

### Approach

1. Load one subject (PSG + hypnogram)
2. Extract EEG channel (Fpz-Cz)
3. Split signal into 30-second epochs
4. Assign one label per epoch
5. Remove invalid epochs
6. Store dataset for further analysis

---

### Note

We start with a single subject to validate the pipeline before scaling to multiple subjects.


In [2]:
import mne
import numpy as np
import pandas as pd
from pathlib import Path

In [3]:
BASE = Path("../data/raw/physionet.org/files/sleep-edfx/1.0.0/sleep-cassette")

psg_file = BASE / "SC4001E0-PSG.edf"
hyp_file = BASE / "SC4001EC-Hypnogram.edf"

print(psg_file.exists(), psg_file.name)
print(hyp_file.exists(), hyp_file.name)

True SC4001E0-PSG.edf
True SC4001EC-Hypnogram.edf


In [4]:
raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)

raw.pick_channels(["EEG Fpz-Cz"])

eeg, times = raw.get_data(return_times=True)
eeg = eeg[0]

sfreq = raw.info["sfreq"]

print("Sampling frequency:", sfreq)
print("EEG shape:", eeg.shape)
print("Duration hours:", times[-1] / 3600)

/var/folders/53/ps9kkpj52j71m9fx4fmfqxsw0000gn/T/ipykernel_53000/2494266797.py:1: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
/var/folders/53/ps9kkpj52j71m9fx4fmfqxsw0000gn/T/ipykernel_53000/2494266797.py:1: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
/var/folders/53/ps9kkpj52j71m9fx4fmfqxsw0000gn/T/ipykernel_53000/2494266797.py:1: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Sampling frequency: 100.0
EEG shape: (7950000,)
Duration hours: 22.083330555555555


In [5]:
annot = mne.read_annotations(hyp_file)

print(annot)
print(set(str(d) for d in annot.description))

<Annotations | 154 segments: Sleep stage 1 (24), Sleep stage 2 (40), Sleep ...>
{'Sleep stage 4', 'Sleep stage 2', 'Sleep stage W', 'Sleep stage 3', 'Sleep stage 1', 'Sleep stage ?', 'Sleep stage R'}


In [6]:
stage_map = {
    "Sleep stage W": 0,   # Wake
    "Sleep stage 1": 1,   # N1
    "Sleep stage 2": 2,   # N2
    "Sleep stage 3": 3,   # N3
    "Sleep stage 4": 3,   # N3, old Rechtschaffen & Kales stage 4
    "Sleep stage R": 4,   # REM
}

stage_names = {
    0: "W",
    1: "N1",
    2: "N2",
    3: "N3",
    4: "REM",
}

In [7]:
EPOCH_SEC = 30
epoch_samples = int(EPOCH_SEC * sfreq)

X = []
y = []
epoch_info = []

subject_id = "SC4001"

for onset, duration, description in zip(
    annot.onset, annot.duration, annot.description
):
    description = str(description)

    if description not in stage_map:
        continue

    stage = stage_map[description]

    n_epochs = int(duration // EPOCH_SEC)

    for k in range(n_epochs):
        start_sec = onset + k * EPOCH_SEC
        end_sec = start_sec + EPOCH_SEC

        start_sample = int(start_sec * sfreq)
        end_sample = start_sample + epoch_samples

        if start_sample < 0:
            continue

        if end_sample > len(eeg):
            continue

        segment = eeg[start_sample:end_sample]

        if len(segment) != epoch_samples:
            continue

        X.append(segment)
        y.append(stage)

        epoch_info.append({
            "subject_id": subject_id,
            "start_sec": start_sec,
            "end_sec": end_sec,
            "stage": stage,
            "stage_name": stage_names[stage],
        })

X = np.array(X)
y = np.array(y)
epoch_df = pd.DataFrame(epoch_info)

print("X shape:", X.shape)
print("y shape:", y.shape)
print(epoch_df.head())

X shape: (2650, 3000)
y shape: (2650,)
  subject_id  start_sec  end_sec  stage stage_name
0     SC4001        0.0     30.0      0          W
1     SC4001       30.0     60.0      0          W
2     SC4001       60.0     90.0      0          W
3     SC4001       90.0    120.0      0          W
4     SC4001      120.0    150.0      0          W


In [8]:
epoch_df["stage_name"].value_counts().sort_index()

stage_name
N1       58
N2      250
N3      220
REM     125
W      1997
Name: count, dtype: int64

In [9]:
non_wake_idx = epoch_df.index[epoch_df["stage"] != 0]

first_sleep_idx = non_wake_idx[0]
last_sleep_idx = non_wake_idx[-1]

buffer_epochs = 60  # 60 epochs × 30 sec = 30 min

keep_start = max(0, first_sleep_idx - buffer_epochs)
keep_end = min(len(epoch_df), last_sleep_idx + buffer_epochs + 1)

X_trim = X[keep_start:keep_end]
y_trim = y[keep_start:keep_end]
epoch_df_trim = epoch_df.iloc[keep_start:keep_end].reset_index(drop=True)

print("Original:", X.shape, y.shape)
print("Trimmed:", X_trim.shape, y_trim.shape)

epoch_df_trim["stage_name"].value_counts().sort_index()

Original: (2650, 3000) (2650,)
Trimmed: (841, 3000) (841,)


stage_name
N1      58
N2     250
N3     220
REM    125
W      188
Name: count, dtype: int64

In [10]:
is_transition = []

for i in range(len(y_trim)):
    if i == 0 or i == len(y_trim) - 1:
        is_transition.append(True)  # edges → treat as transition
        continue

    prev_stage = y_trim[i - 1]
    curr_stage = y_trim[i]
    next_stage = y_trim[i + 1]

    if prev_stage == curr_stage == next_stage:
        is_transition.append(False)
    else:
        is_transition.append(True)

epoch_df_trim["is_transition"] = is_transition

epoch_df_trim.head()

,subject_id,start_sec,end_sec,stage,stage_name,is_transition
0,SC4001,28830.0,28860.0,0,W,True
1,SC4001,28860.0,28890.0,0,W,False
2,SC4001,28890.0,28920.0,0,W,False
3,SC4001,28920.0,28950.0,0,W,False
4,SC4001,28950.0,28980.0,0,W,False


In [11]:
epoch_df_trim["is_transition"].value_counts()

is_transition
False    659
True     182
Name: count, dtype: int64

In [22]:
from scipy.signal import welch

sfreq = raw.info["sfreq"]

def band_power(signal, sfreq, band):
    fmin, fmax = band
    freqs, psd = welch(signal, sfreq, nperseg=256)
    mask = (freqs >= fmin) & (freqs <= fmax)
    return np.trapz(psd[mask], freqs[mask])

def extract_features(signal, sfreq):
    return {
        "mean": np.mean(signal),
        "std": np.std(signal),
        "min": np.min(signal),
        "max": np.max(signal),
        "rms": np.sqrt(np.mean(signal**2)),
        "zero_crossings": np.sum(np.diff(np.sign(signal)) != 0),

        # NEW — frequency features
        "delta": band_power(signal, sfreq, (0.5, 4)),
        "theta": band_power(signal, sfreq, (4, 8)),
        "alpha": band_power(signal, sfreq, (8, 12)),
        "beta":  band_power(signal, sfreq, (12, 30)),
    }

feature_rows = []

for i, segment in enumerate(X_trim):
    feats = extract_features(segment, sfreq)
    feats["stage"] = y_trim[i]
    feats["is_transition"] = epoch_df_trim.loc[i, "is_transition"]
    feature_rows.append(feats)

feature_df = pd.DataFrame(feature_rows)

feature_df.head()

/var/folders/53/ps9kkpj52j71m9fx4fmfqxsw0000gn/T/ipykernel_53000/1167900490.py:9: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(psd[mask], freqs[mask])


,mean,std,min,max,rms,zero_crossings,delta,theta,alpha,beta,stage,is_transition
0,1.862955e-08,0.000021,-0.000073,0.000068,0.000021,451,3.363453e-10,2.730077e-11,3.720581e-12,1.124198e-11,0,True
1,8.478320e-07,0.000026,-0.000088,0.000097,0.000026,421,4.108288e-10,3.697209e-11,3.494540e-12,1.054644e-11,0,False
2,-2.127394e-07,0.000032,-0.000123,0.000102,0.000032,240,6.415054e-10,4.662177e-11,4.109897e-12,8.692037e-12,0,False
3,-3.318935e-07,0.000027,-0.000081,0.000089,0.000027,392,4.859145e-10,3.712763e-11,4.679157e-12,1.744431e-11,0,False
4,3.947214e-07,0.000029,-0.000091,0.000083,0.000029,320,3.783164e-10,3.323889e-11,4.063267e-12,1.044153e-11,0,False


In [28]:
feature_cols = [
    "mean",
    "std",
    "min",
    "max",
    "rms",
    "zero_crossings",
    "delta",
    "theta",
    "alpha",
    "beta",
]

X_feat = feature_df[feature_cols]
y_stage = feature_df["stage"]

print(X_feat.shape)
print(y_stage.value_counts().sort_index())

(841, 10)
stage
0    188
1     58
2    250
3    220
4    125
Name: count, dtype: int64


In [29]:
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_feat,
    y_stage,
    feature_df.index,
    test_size=0.25,
    random_state=42,
    stratify=y_stage
)

In [30]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Overall accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(
    y_test,
    y_pred,
    target_names=["W", "N1", "N2", "N3", "REM"]
))

Overall accuracy: 0.8199052132701422
              precision    recall  f1-score   support

           W       0.79      0.89      0.84        47
          N1       0.75      0.20      0.32        15
          N2       0.88      0.81      0.84        63
          N3       0.93      0.93      0.93        55
         REM       0.63      0.84      0.72        31

    accuracy                           0.82       211
   macro avg       0.80      0.73      0.73       211
weighted avg       0.83      0.82      0.81       211



In [31]:
test_meta = feature_df.loc[idx_test].copy()
test_meta["y_true"] = y_test.values
test_meta["y_pred"] = y_pred

stable_mask = test_meta["is_transition"] == False
transition_mask = test_meta["is_transition"] == True

stable_acc = accuracy_score(
    test_meta.loc[stable_mask, "y_true"],
    test_meta.loc[stable_mask, "y_pred"]
)

transition_acc = accuracy_score(
    test_meta.loc[transition_mask, "y_true"],
    test_meta.loc[transition_mask, "y_pred"]
)

print("Stable accuracy:", stable_acc)
print("Transition accuracy:", transition_acc)

print("Stable samples:", stable_mask.sum())
print("Transition samples:", transition_mask.sum())

Stable accuracy: 0.8682634730538922
Transition accuracy: 0.6363636363636364
Stable samples: 167
Transition samples: 44


In [32]:
print("Stage distribution (stable):")
print(test_meta.loc[stable_mask, "y_true"].value_counts())

print("\nStage distribution (transition):")
print(test_meta.loc[transition_mask, "y_true"].value_counts())

Stage distribution (stable):
y_true
2    48
3    44
0    43
4    26
1     6
Name: count, dtype: int64

Stage distribution (transition):
y_true
2    15
3    11
1     9
4     5
0     4
Name: count, dtype: int64


In [33]:
from sklearn.metrics import confusion_matrix

cm_stable = confusion_matrix(
    test_meta.loc[stable_mask, "y_true"],
    test_meta.loc[stable_mask, "y_pred"]
)

cm_transition = confusion_matrix(
    test_meta.loc[transition_mask, "y_true"],
    test_meta.loc[transition_mask, "y_pred"]
)

print("Stable confusion matrix:\n", cm_stable)
print("\nTransition confusion matrix:\n", cm_transition)

Stable confusion matrix:
 [[38  1  1  1  2]
 [ 1  1  0  0  4]
 [ 4  0 40  0  4]
 [ 0  0  2 42  0]
 [ 0  0  2  0 24]]

Transition confusion matrix:
 [[ 4  0  0  0  0]
 [ 3  2  0  0  4]
 [ 0  0 11  3  1]
 [ 2  0  0  9  0]
 [ 1  0  2  0  2]]


## All Subjects

In [61]:
def process_subject(subject_id, BASE):
    psg_file = BASE / f"{subject_id}E0-PSG.edf"
    hyp_file = BASE / f"{subject_id}EC-Hypnogram.edf"

    raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
    raw.pick_channels(["EEG Fpz-Cz"])

    eeg, times = raw.get_data(return_times=True)
    eeg = eeg[0]
    sfreq = raw.info["sfreq"]

    annot = mne.read_annotations(hyp_file)

    # --- Stage mapping ---
    stage_map = {
        "Sleep stage W": 0,
        "Sleep stage 1": 1,
        "Sleep stage 2": 2,
        "Sleep stage 3": 3,
        "Sleep stage 4": 3,
        "Sleep stage R": 4,
    }

    EPOCH_SEC = 30
    epoch_samples = int(EPOCH_SEC * sfreq)

    X = []
    y = []

    for onset, duration, description in zip(
        annot.onset, annot.duration, annot.description
    ):
        description = str(description)

        if description not in stage_map:
            continue

        stage = stage_map[description]
        n_epochs = int(duration // EPOCH_SEC)

        for k in range(n_epochs):
            start_sec = onset + k * EPOCH_SEC
            start_sample = int(start_sec * sfreq)
            end_sample = start_sample + epoch_samples

            if end_sample > len(eeg):
                continue

            segment = eeg[start_sample:end_sample]

            if len(segment) != epoch_samples:
                continue

            X.append(segment)
            y.append(stage)

    X = np.array(X)
    y = np.array(y)

    # --- Trim wake ---
    non_wake_idx = np.where(y != 0)[0]
    if len(non_wake_idx) == 0:
        return None

    buffer_epochs = 60
    start = max(0, non_wake_idx[0] - buffer_epochs)
    end = min(len(y), non_wake_idx[-1] + buffer_epochs)

    X = X[start:end]
    y = y[start:end]

    # --- Transition label ---
    is_transition = []
    for i in range(len(y)):
        if i == 0 or i == len(y) - 1:
            is_transition.append(True)
            continue
        if y[i-1] == y[i] == y[i+1]:
            is_transition.append(False)
        else:
            is_transition.append(True)

    # --- Features ---
    def band_power(signal, sfreq, band):
        fmin, fmax = band
        freqs, psd = welch(signal, sfreq, nperseg=256)
        mask = (freqs >= fmin) & (freqs <= fmax)
        return np.trapz(psd[mask], freqs[mask])

    def extract_features(signal):
        return {
            "mean": np.mean(signal),
            "std": np.std(signal),
            "min": np.min(signal),
            "max": np.max(signal),
            "rms": np.sqrt(np.mean(signal**2)),
            "zero_crossings": np.sum(np.diff(np.sign(signal)) != 0),
            "delta": band_power(signal, sfreq, (0.5, 4)),
            "theta": band_power(signal, sfreq, (4, 8)),
            "alpha": band_power(signal, sfreq, (8, 12)),
            "beta":  band_power(signal, sfreq, (12, 30)),
        }

    rows = []
    for i, segment in enumerate(X):
        segment = (segment - np.mean(segment)) / (np.std(segment) + 1e-8)
        feats = extract_features(segment)
        feats["stage"] = y[i]
        feats["is_transition"] = is_transition[i]
        feats["subject_id"] = subject_id
        rows.append(feats)

    return pd.DataFrame(rows)

In [62]:
import re

psg_files = list(BASE.glob("*PSG.edf"))

valid_subjects = []

for psg in psg_files:
    subject_id = psg.name.split("E0")[0]

    hyp_file = BASE / f"{subject_id}EC-Hypnogram.edf"

    if hyp_file.exists():
        valid_subjects.append(subject_id)
    else:
        print("Missing hypnogram for:", subject_id)

print("Valid subjects:", valid_subjects)

Missing hypnogram for: SC4022
Missing hypnogram for: SC4011
Missing hypnogram for: SC4021
Missing hypnogram for: SC4032
Valid subjects: ['SC4002', 'SC4031', 'SC4041', 'SC4012', 'SC4001']


In [63]:
valid_subjects = ['SC4002', 'SC4031', 'SC4041', 'SC4012', 'SC4001']

all_dfs = []

for subject in valid_subjects:
    print("Processing:", subject)
    df = process_subject(subject, BASE)

    if df is not None:
        all_dfs.append(df)

feature_df_all = pd.concat(all_dfs, ignore_index=True)

print(feature_df_all.shape)
feature_df_all["subject_id"].value_counts()

Processing: SC4002


/var/folders/53/ps9kkpj52j71m9fx4fmfqxsw0000gn/T/ipykernel_53000/188745941.py:5: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
/var/folders/53/ps9kkpj52j71m9fx4fmfqxsw0000gn/T/ipykernel_53000/188745941.py:5: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
/var/folders/53/ps9kkpj52j71m9fx4fmfqxsw0000gn/T/ipykernel_53000/188745941.py:5: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/var/folders/53/ps9kkpj52j71m9fx4fmfqxsw0000gn/T/ipykernel_53000/188745941.py:88: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(psd[mask], freqs[mask])


Processing: SC4031


...

/var/folders/53/ps9kkpj52j71m9fx4fmfqxsw0000gn/T/ipykernel_53000/188745941.py:5: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
/var/folders/53/ps9kkpj52j71m9fx4fmfqxsw0000gn/T/ipykernel_53000/188745941.py:5: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
/var/folders/53/ps9kkpj52j71m9fx4fmfqxsw0000gn/T/ipykernel_53000/188745941.py:5: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Processing: SC4001


/var/folders/53/ps9kkpj52j71m9fx4fmfqxsw0000gn/T/ipykernel_53000/188745941.py:5: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
/var/folders/53/ps9kkpj52j71m9fx4fmfqxsw0000gn/T/ipykernel_53000/188745941.py:5: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
/var/folders/53/ps9kkpj52j71m9fx4fmfqxsw0000gn/T/ipykernel_53000/188745941.py:5: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
(5336, 13)


subject_id
SC4041    1234
SC4012    1185
SC4002    1126
SC4031     951
SC4001     840
Name: count, dtype: int64

In [64]:
feature_cols = [c for c in feature_df_all.columns 
                if c not in ["stage", "is_transition", "subject_id"]]

X_feat = feature_df_all[feature_cols]
y_stage = feature_df_all["stage"]

print(X_feat.shape)
print(y_stage.value_counts().sort_index())

(5336, 10)
stage
0     868
1     436
2    2388
3     723
4     921
Name: count, dtype: int64


In [65]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)

groups = feature_df_all["subject_id"]

for train_idx, test_idx in gss.split(X_feat, y_stage, groups):
    X_train = X_feat.iloc[train_idx]
    X_test = X_feat.iloc[test_idx]
    y_train = y_stage.iloc[train_idx]
    y_test = y_stage.iloc[test_idx]

    idx_train = train_idx
    idx_test = test_idx

In [66]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Overall accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["W", "N1", "N2", "N3", "REM"]
))

Overall accuracy: 0.6745762711864407
              precision    recall  f1-score   support

           W       0.50      0.94      0.65       381
          N1       0.25      0.10      0.14       225
          N2       0.79      0.85      0.82       993
          N3       0.93      0.75      0.83       350
         REM       0.54      0.26      0.35       411

    accuracy                           0.67      2360
   macro avg       0.60      0.58      0.56      2360
weighted avg       0.67      0.67      0.65      2360



In [68]:
test_meta = feature_df_all.iloc[idx_test].copy()
test_meta["y_true"] = y_test.values
test_meta["y_pred"] = y_pred

stable_mask = test_meta["is_transition"] == False
transition_mask = test_meta["is_transition"] == True

stable_acc = accuracy_score(
    test_meta.loc[stable_mask, "y_true"],
    test_meta.loc[stable_mask, "y_pred"]
)

transition_acc = accuracy_score(
    test_meta.loc[transition_mask, "y_true"],
    test_meta.loc[transition_mask, "y_pred"]
)

print("Stable accuracy:", stable_acc)
print("Transition accuracy:", transition_acc)
print("Stable samples:", stable_mask.sum())
print("Transition samples:", transition_mask.sum())

Stable accuracy: 0.7164811870694223
Transition accuracy: 0.507399577167019
Stable samples: 1887
Transition samples: 473


In [60]:
# summary_no_norm = pd.DataFrame([{
#     "stable_accuracy": stable_acc,
#     "transition_accuracy": transition_acc,
#     "gap": stable_acc - transition_acc,
# }])

# summary_no_norm.to_csv(
#     PROCESSED_DIR / "baseline_no_normalization.csv",
#     index=False
# )

In [50]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

feature_df_all.to_parquet(
    PROCESSED_DIR / "sleep_epoch_features.parquet",
    index=False
)

print("Saved:", PROCESSED_DIR / "sleep_epoch_features.parquet")
print(feature_df_all.shape)

Saved: ../data/processed/sleep_epoch_features.parquet
(5336, 13)


In [51]:
test_meta.to_parquet(
    PROCESSED_DIR / "sleep_transition_test_results.parquet",
    index=False
)

print("Saved:", PROCESSED_DIR / "sleep_transition_test_results.parquet")
print(test_meta.shape)

Saved: ../data/processed/sleep_transition_test_results.parquet
(2360, 15)


In [52]:
summary = pd.DataFrame([{
    "stable_accuracy": stable_acc,
    "transition_accuracy": transition_acc,
    "gap": stable_acc - transition_acc,
    "stable_samples": stable_mask.sum(),
    "transition_samples": transition_mask.sum(),
    "n_subjects": feature_df_all["subject_id"].nunique(),
    "n_epochs": len(feature_df_all),
}])

summary.to_csv(
    PROCESSED_DIR / "baseline_transition_summary.csv",
    index=False
)

summary

,stable_accuracy,transition_accuracy,gap,stable_samples,transition_samples,n_subjects,n_epochs
0,0.716481,0.5074,0.209082,1887,473,5,5336


## Summary of Dataset Construction and Baseline Check

This notebook created a multi-subject epoch-level dataset from Sleep-EDF EEG recordings.

Each row represents one 30-second EEG epoch with:

- sleep stage label
- transition/stable label
- basic time-domain features
- frequency-band power features
- subject identifier

A baseline Random Forest model was used as a sanity check.

The key result was that classification accuracy was higher for stable epochs than for transition epochs, even after per-epoch normalization.

This suggests that sleep-stage inference is not uniformly difficult. Errors concentrate near stage transitions, where the physiological signal may represent a mixed or ambiguous state rather than a clean discrete category.